# SGDense-3DGS — experiment runner

Runs the experiment manifest (upstream **3DGS baseline** + **SGDense**) and
extracts metrics, using the `experiments/` harness. The harness handles
cloning external codebases, installing the right CUDA rasterizer, and
**swapping rasterizers** between methods (SGDense seg-aware fork vs upstream
stock). See `experiments/README.md` for details.


## 1. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


## 2. Get this repo + dependencies

Clones/refreshes the SGDense-3DGS fork (which contains `train.py`, the
`experiments/` harness, and the `eval_sgdense.py` evaluator) and initializes
the two training submodules. The CUDA rasterizers themselves are built by the
harness per method (step 6).


In [ ]:
import os

REPO_URL = "https://github.com/satyaba/gaussian-splatting.git"
REPO_DIR = "/content/gaussian-splatting"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch --all -q
    !git -C {REPO_DIR} pull --ff-only -q

os.chdir(REPO_DIR)

# Training submodules: seg-aware rasterizer + simple-knn.
!git submodule update --init --recursive submodules/diff-gaussian-rasterization submodules/simple-knn
# MUST print 813db1d (the kernel seg-vote path); if not, the clone is stale.
!git -C submodules/diff-gaussian-rasterization log --oneline -1

print("cwd:", os.getcwd())


In [ ]:
!pip install -q plyfile opencv-python tqdm joblib lpips torchmetrics


## 3. GPU / CUDA check

The rasterizer is compiled against a CUDA toolkit that must match
`torch.version.cuda`. Known-good Colab runtime: **2026.10**.


In [ ]:
!nvidia-smi

import re, shutil, subprocess, warnings
import torch

print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | "
          f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

PINNED_RASTERIZER_CUDA_MAJOR = "12"

def _nvcc_cuda():
    exe = shutil.which("nvcc")
    if exe is None:
        return None
    out = subprocess.run([exe, "--version"], capture_output=True, text=True).stdout
    m = re.search(r"release (\d+)\.(\d+)", out)
    return f"{m.group(1)}.{m.group(2)}" if m else None

torch_cuda, nvcc_cuda = torch.version.cuda, _nvcc_cuda()
print(f"  torch built for CUDA {torch_cuda} | nvcc {nvcc_cuda or 'NOT FOUND'}")

if nvcc_cuda is None:
    warnings.warn("nvcc not found — CUDA extensions cannot be built. Use runtime 2026.10.")
elif torch_cuda is None:
    warnings.warn("This PyTorch build has no CUDA; extensions cannot be built.")
elif torch_cuda.split(".")[0] != nvcc_cuda.split(".")[0]:
    raise RuntimeError(f"CUDA major mismatch: torch {torch_cuda} vs nvcc {nvcc_cuda}.")
elif torch_cuda.split(".")[0] != PINNED_RASTERIZER_CUDA_MAJOR:
    warnings.warn(f"CUDA {torch_cuda} != pinned {PINNED_RASTERIZER_CUDA_MAJOR}.x; build may fail.")
else:
    print("  OK: toolchain consistent with the pinned rasterizer.")


## 4. Configure paths

`experiments/runs.py` assumes `<HOME_DIR>/Dataset/<dataset>` with
`images/`, `sparse/0/`, and (for SGDense) `semantic_class/`.


In [ ]:
import os
from pathlib import Path

HOME_DIR = "/content/drive/MyDrive/Magister/Thesis/Script"
DATASET_NAME = "office_0/Sequence_1"

os.environ["HOME_DIR"] = HOME_DIR
os.environ["DATASET_ROOT"] = str(Path(HOME_DIR) / "Dataset")

DATASET_DIR = Path(os.environ["DATASET_ROOT"]) / DATASET_NAME
os.environ["DATASET_DIR"] = str(DATASET_DIR)
os.environ["SEG_DIR"] = str(DATASET_DIR / "semantic_class")

for k in ("HOME_DIR", "DATASET_ROOT", "DATASET_DIR", "SEG_DIR"):
    print(f"{k} = {os.environ[k]}")


## 5. Inspect the run manifest


In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())          # make `experiments` importable from repo root

from experiments.runs import RUNS, run_by_id, run_all, run_sanity

for r in RUNS.values():
    print(f"{r.run_id:14s} method={r.method:12s} iters={r.iterations:<6d} {r.notes}")


## 6. Sanity check (run this first)

Runs the two short 3 000-iteration versions (`vanilla_sanity`, `sgdense_sanity`)
and extracts their metrics. 3 000 iters crosses the seg warmup (2000), runs
densification with floater/freeze handling, hits the opacity reset (3000) and
saves — so it exercises the whole pipeline (train → rasterizer swap → eval)
before the full 30 000-iteration runs.

Preview without executing: `run_sanity(dry=True)`.


In [ ]:
# Short end-to-end smoke test (also builds/swaps rasterizers under the hood):
run_sanity()


In [ ]:
from experiments.metrics import extract_sanity

extract_sanity()


## 7. Full training (30k)

`run_all()` trains every non-sanity run. If you already ran the sanity check,
the correct rasterizer for the next method is likely cached (no rebuild).
Run one at a time with `run_by_id(...)`.


In [ ]:
run_all()

# --- or one at a time ---
# run_by_id("vanilla_30k")
# run_by_id("sgdense_30k")

# --- per-run hyperparameter overrides (appended to the spec's args) ---
# run_by_id("sgdense_photo20k_pos_30k", extra="--boundary_weight 0.2 --interior_weight 0.4")
# run_by_id("sgdense_photo20k_tex_grad_30k", extra=["--texture_beta", "0.5"])

# --- a sweep: tag gives each call its own output dir ---
# for bw in (0.5, 0.3, 0.1):
#     run_by_id("sgdense_photo20k_pos_30k", tag=f"bw{bw}", extra=f"--boundary_weight {bw}")

# --- or preview the commands only ---
# run_all(dry=True)


## 8. Extract metrics

Computes train/test PSNR / SSIM / LPIPS + Gaussian count + timings and
appends a row per run to `$HOME_DIR/output/metrics.csv` (same code path for
SGDense and vanilla models).


In [ ]:
from experiments.metrics import extract_all, extract_run

extract_all()
# extract_run("sgdense_30k")


## 9. Results


In [ ]:
import os
import pandas as pd

csv_path = os.path.join(os.environ["HOME_DIR"], "output", "metrics.csv")
print(csv_path)
pd.read_csv(csv_path)


## 10. (Optional) Ad-hoc evaluation

Score an arbitrary model dir, e.g. one of your earlier 12k runs:

```python
from experiments.metrics import extract_one
extract_one('$HOME_DIR/output/sgdense', iteration=12000, name='sgdense@12k',
            run_id='sgdense_12k')
```
